# Risk scoring + Explainability + Random Forest

Owner: Thanthrige D.P

Put `PhiUSIIL_Phishing_URL_Dataset.csv` in `data/raw/` before running.

**How this notebook is organised (2026-11-24):**
- **Part A — updated protocol (use this for the progress evaluation).** Group-aware split, grouped 5-fold CV,
  primary feature set `url_engineered` (URL-only, no `URLSimilarityIndex`, no `IsHTTPS`, no page content), the Random
  Forest frozen from a bounded grouped search, grouped calibration, risk bands and SHAP on held-out URLs. Run the cells
  from *Setup* to the end of Part A (about 5 minutes on the 20k sample).
- **Part B — historical record (phases 1–5, 17–19 Nov).** The original work on the ungrouped random split with all
  50 original features (including page content). Kept unchanged as the evidence trail of the leakage investigation
  that led to the group decisions; **its numbers are not the official results** and it emits `UngroupedCVWarning`.

## To do

- [x] Train, tune and evaluate the Random Forest in `src/models/forest.py` (bounded grouped search on a training subset, Part A)
- [x] Risk score from the phishing probability, risk bands (4.11)
- [x] Calibration check: calibration curve / Brier score before and after (grouped folds in Part A)
- [x] SHAP global importance and summary plot (4.13)
- [x] Explain single predictions: score + class + top features (`demo.py` does the same from the saved models)
- [x] Full-data run and final test-set report (`python main.py` → `reports/official_comparison.csv`, `reports/risk_bands.csv`)
- [ ] Final analysis across all parts (4.14): after the progress evaluation

## Setup

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import matplotlib.pyplot as plt

from src import config, data, features, evaluate, risk, explain
from src.models import logreg, tree, svm, forest

# Work on a sample first; set SAMPLE = None for the full dataset.
SAMPLE = 20000

df = data.load_clean_data()
if SAMPLE and SAMPLE < len(df):
    df = df.sample(n=SAMPLE, random_state=config.RANDOM_STATE).reset_index(drop=True)
df.shape


## Part A — updated protocol (2026-11-24)

1. **Rows:** `data.split_index()` (group-aware; near-duplicate URLs never cross train/test).
2. **Features:** `features.build_feature_sets()` on training rows only; the forest uses the primary `url_engineered` set.
3. **Hyperparameters:** frozen in `forest.BEST_PARAMS` from a *bounded* grouped search (`tune_models.py` part
   `rf_subset`): a reproducible group-preserving subset of the training rows (`data.group_subset`), the same grouped
   5-fold CV, 8 candidates, and a one-SE rule preferring the forest with fewer leaves per tree. Those CV numbers come
   from the **subset** and are tuning evidence only; the final forest is trained on the full training split by `main.py`.
4. **Risk score:** sigmoid-calibrated forest probability (`risk.calibrate`, grouped folds on training rows only);
   bands Low < 0.30 ≤ Medium < 0.70 ≤ High.
5. **Explanations:** SHAP of the uncalibrated forest on a few hundred held-out URLs (explanations choose nothing).


In [ ]:

import warnings
warnings.filterwarnings("ignore", message=".*sklearn.utils.parallel.delayed.*")  # harmless joblib notice

PRIMARY = features.PRIMARY_FEATURE_SET
dfa = features.add_features(df)
groups_a = data.group_ids(dfa)
train_a, test_a = data.split_index(dfa)
sets_a = features.build_feature_sets(dfa.loc[train_a])            # training rows only
cols_a = sets_a[PRIMARY]
Xa_train, Xa_test, ya_train, ya_test = data.split(dfa, cols_a)
ga_train = groups_a.loc[Xa_train.index]
assert not set(ga_train) & set(groups_a.loc[Xa_test.index])
print(f"{PRIMARY}: {len(cols_a)} features | train {len(Xa_train):,} | test {len(Xa_test):,} | SAMPLE = {SAMPLE}")
print("forest.build():", forest.BEST_PARAMS, "|", forest.TUNING_NOTE)


In [ ]:

# Stored full-data tuning evidence (group-preserving training subset); not rerun here.
rf_search = pd.read_csv(config.RESULTS_DIR / "tuning" / "rf_subset_search.csv")
show = ["max_depth", "min_samples_leaf", "max_features", "n_estimators", "f1_mean", "f1_std", "f1_se",
        "roc_auc_mean", "precision_mean", "recall_mean", "leaves_per_tree_mean", "depth_per_tree_mean",
        "within_1se_of_best", "chosen"]
display(rf_search[show].round(5))
import json
summary = json.loads((config.RESULTS_DIR / "tuning" / "tuning_summary.json").read_text())["rf_subset"]
print({k: summary[k] for k in ("subset_rows", "subset_groups", "subset_phishing_share", "complete", "seconds")})


In [ ]:

# Grouped CV of the frozen forest on this run's training rows (sample: development check only).
rf_cv = evaluate.cv_evaluate(forest.build(), Xa_train, ya_train, groups=ga_train)
pd.Series(rf_cv).round(5)



### Risk score calibration and bands (grouped, training rows only)

`risk.calibration_report(..., groups=)` scores none / sigmoid / isotonic **out-of-fold** on the grouped folds.
(The calibration *inside* each outer fold uses the legacy ungrouped inner splitter, see `risk._maybe_calibrated`;
the outer validation rows are never used for fitting.) The served model, `risk.calibrate`, uses grouped folds.


In [ ]:

calib_a, oof_a = risk.calibration_report(forest.build(), Xa_train, ya_train, groups=ga_train)
display(calib_a.round(5))
suffix = f"_sample_{SAMPLE}" if SAMPLE else ""
fig = risk.plot_calibration(ya_train, oof_a, filename=f"rf_calibration_curve_{PRIMARY}{suffix}.png",
                            title=f"Random Forest calibration ({PRIMARY}, grouped out-of-fold, training rows)")
plt.show()
pd.concat({m: risk.band_table(ya_train, s).set_index("band") for m, s in oof_a.items()}).round(4)


In [ ]:

# SHAP on held-out URLs (explanations only) and the served, calibrated risk score.
rf_a = forest.train(Xa_train, ya_train)
X_explain_a = Xa_test.iloc[:300]
shap_a = explain.contributions(rf_a, X_explain_a)
imp_a = explain.rank_importance(X_explain_a.columns, shap_a.values, "mean_abs_shap")
explain.summary_plot(rf_a, X_explain_a, f"rf_shap_summary_{PRIMARY}{suffix}.png", values=shap_a.values)
display(imp_a.head(12).round(4))

risk_a = risk.calibrate(forest.build(), Xa_train, ya_train, groups=ga_train)
scores_a = risk.risk_score(risk_a, X_explain_a)
display(risk.band_table(ya_test.loc[X_explain_a.index], scores_a).round(4))


In [ ]:

per_url_a = explain.explain_predictions(rf_a, X_explain_a, scores_a, top=3, contrib=shap_a)
per_url_a.insert(0, "URL", dfa.loc[X_explain_a.index, "URL"].values)
per_url_a.insert(1, "is_phishing", ya_test.loc[X_explain_a.index].values)
per_url_a.insert(2, "bare_https_www", data.bare_homepage_mask(dfa.loc[X_explain_a.index, "URL"]).values)
examples_a = {
    "clear phishing": per_url_a[per_url_a["is_phishing"] == 1]["risk_score"].idxmax(),
    "clear legitimate": per_url_a[per_url_a["is_phishing"] == 0]["risk_score"].idxmin(),
    "most uncertain": (per_url_a["risk_score"] - 0.5).abs().idxmin(),
}
for name, i in examples_a.items():
    ex = explain.explain_one(rf_a, X_explain_a.loc[[i]], scores_a[X_explain_a.index.get_loc(i)], top=5)
    truth = "phishing" if per_url_a.loc[i, "is_phishing"] else "legitimate"
    print(f"{name}: {per_url_a.loc[i, 'URL']}  (true: {truth}; bare https://www homepage: {per_url_a.loc[i, 'bare_https_www']})")
    print(f"  risk score {ex.attrs['risk_score']:.3f} -> {ex.attrs['risk_level']}")
    display(ex.round(4))


### Part A findings (full-data numbers from `python main.py`, 2026-11-24)

Source files: `reports/official_comparison.csv`, `reports/format_baseline.csv`, `reports/format_controlled.csv`,
`reports/risk_bands.csv`, `reports/shap_importance.csv` (tables: `docs/progress_evaluation/results_tables.md`).

- **Random Forest, `url_engineered`:** grouped CV F1 0.9950 ± 0.0002 on the full training split (the subset search
  gave 0.9912 on 50,000 rows: more data helps the forest); held-out F1 0.9950, ROC-AUC 0.9988, TN 26,894 / FP 75 /
  FN 125 / TP 19,979.
- **Against the format rule** ("not a bare https://www homepage → phishing", no learning): the rule has CV F1 0.9948
  and test F1 0.9948, so the forest only **matches** the collection-format baseline overall. On the 27,178 held-out
  bare-homepage URLs (where the format cue is absent) it finds 104 of 209 phishing URLs with 75 false alarms; the rule
  finds none.
- **Risk score:** sigmoid-calibrated (grouped folds, training rows), test Brier 0.0038. Bands: Low 57 % of URLs
  (0.4 % phishing), Medium 0.2 % (40 % phishing), High 43 % (99.7 % phishing). **108 held-out phishing URLs fall in Low
  and 61 legitimate URLs in High**, so "Low" means low estimated risk, not "safe". These 108 Low-band phishing URLs
  are a calibrated-forest band count, not Logistic Regression's 108 false negatives: the counts are equal by
  coincidence and the two sets only partly overlap (`docs/progress_evaluation/build/check_low_band.py`).
- **Calibration choice re-checked on a 20,000-row development sample** (the `calibration-grouped` cell, 16,000
  training rows of the sample, 2026-11-24; re-run and reproduced the same day): out-of-fold Brier / log loss: none
  0.0131 / 0.0618, **sigmoid 0.0109 / 0.0432**, isotonic 0.0109 / 0.0476, ROC-AUC unchanged (0.998), so
  `risk.CALIBRATION = "sigmoid"` stays. Caveat: the outer folds were grouped, but the **internal calibration CV folds
  were ungrouped** (`risk._maybe_calibrated` uses the legacy splitter), and the check was not repeated on the full
  training split. The served model (`risk.calibrate`) uses grouped folds. Trade-off (open group decision): calibration moves more
  phishing URLs into Low (74 vs 40 of 6,834 training-row phishing URLs on the sample) while shrinking Medium.
- **SHAP:** the forest relies on URL character composition (`LetterRatioInURL`, `SpacialCharRatioInURL`,
  `NoOfOtherSpecialCharsInURL`, `DomainLength`, `URLLength`, digit counts). These are exactly the features that encode
  "bare homepage vs URL with path/query", so the explanations are consistent with the format bias rather than evidence
  against it. No page-content feature is used in the primary set (in Part B they dominated, 74 % of |SHAP|).
- **Demo:** `python demo.py` prints score, band, prediction, true label, bare-homepage flag and top SHAP contributions
  for held-out URLs (`reports/demo_full_2026-11-24.txt`).


---
# Part B — historical record (phases 1–5, 17–19 Nov 2026)

Everything below was run on the **ungrouped random split** with the **50 original features** (including page
content and, before phase 3, `URLSimilarityIndex`). It is kept unchanged because it documents how the leakage
and collection-bias problems were found. **Do not quote its numbers as results.** It takes about 15 minutes on the
20k sample and emits `UngroupedCVWarning`.


## Phase 1: Random Forest (proposal 4.9)

Cross-validation on the training set only (`evaluate.cv_splitter()`, 5 stratified folds).
The test set is not touched here.

In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*sklearn.utils.parallel.delayed.*")  # harmless joblib notice
from sklearn.ensemble import RandomForestClassifier

original = data.get_original_features(df)
df = features.add_features(df)
candidates = original + features.ENGINEERED_FEATURES
X_train_all, _, _, _ = data.split(df, candidates)
# Feature selection sees training rows only.
engineered = features.select_features(df.loc[X_train_all.index], candidates)
feature_sets = {"original": original, "engineered": engineered}
{name: len(cols) for name, cols in feature_sets.items()}

In [ ]:
SHOW = ["f1_mean", "f1_std", "roc_auc_mean", "recall_mean", "precision_mean", "accuracy_mean"]

def cv_table(make_model):
    """CV metrics of make_model() on each feature set (training rows only)."""
    rows = {}
    for name, cols in feature_sets.items():
        X_tr, _, y_tr, _ = data.split(df, cols)
        rows[name] = evaluate.cv_evaluate(make_model(), X_tr, y_tr)
    return pd.DataFrame(rows).T[SHOW]

def default_forest():
    """Baseline: sklearn defaults with 200 trees (the starting build())."""
    return RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=config.RANDOM_STATE)

baseline_cv = cv_table(default_forest)
baseline_cv.round(4)

### Hyperparameter search

`forest.tune()`: RandomizedSearchCV, 20 candidates, shared CV folds, chosen by F1,
on the original-feature training set. One set of values goes into `build()`,
because `main.py` uses the same `build()` for both feature sets.

In [ ]:
X_train, X_test, y_train, y_test = data.split(df, original)
search, seconds = forest.tune(X_train, y_train, n_iter=20, scoring="f1")
print(f"Search took {seconds/60:.1f} min for {len(search.cv_results_['params'])} candidates x {config.CV_FOLDS} folds")

# Full search results to a file; only the top rows are shown here.
out = config.RESULTS_DIR / (f"results_sample_{SAMPLE}_forest_search.csv" if SAMPLE else "forest_search.csv")
pd.DataFrame(search.cv_results_).to_csv(out, index=False)
print("best:", search.best_params_)
forest.search_table(search, top=8)

### Tuned model vs baseline

The search is flat: six candidates tie for best CV F1 (0.99993, std 0.00015), and every
other candidate is one URL worse over all folds, which is within noise. `best_params_`
just reports the first of the tied six. Tie-break rule: take the cheapest and most
regularised tied candidate (100 trees, max_depth 30, min_samples_leaf 8, sqrt features),
since extra trees buy nothing here and larger leaves give smoother probabilities for the
risk score. These values are written into `forest.build()`. Provisional until the
leakage check (phase 2): scores this close to 1.0 are a warning, not a result.


In [ ]:
tuned_cv = cv_table(forest.build)
comparison = pd.concat({"baseline": baseline_cv, "tuned": tuned_cv}).round(4)
comparison

## Phase 2: Leakage check

CV F1 of 0.9999 (1 error in 16,000 URLs) is a warning, not a result. This section looks for
features that give the label away, using **training rows only** (`X_train`, `y_train` from the
original-feature split above). The test set is still untouched.

Three checks:
1. **Single-feature ROC-AUC.** Each feature is used directly as a score, with no model fitted.
   `sep_auc = max(AUC, 1 - AUC)`, so 1.0 means that feature alone separates the classes perfectly.
2. **Class-constant features.** These take one value for every URL in one class, which points to
   how the dataset was collected rather than to anything about phishing.
3. **Ablation.** CV of the tuned forest after removing suspect features or feature groups.

This section only measures. The group decides what to drop (`features.select_features()`, owner Madushan S.C).

In [ ]:
from sklearn.metrics import roc_auc_score

def single_feature_auc(X, y):
    """ROC-AUC of each column used directly as a score (no fitting), sorted by separability."""
    rows = []
    for c in X.columns:
        auc = roc_auc_score(y, X[c])
        rows.append({"feature": c, "auc": auc, "sep_auc": max(auc, 1 - auc),
                     "n_unique": X[c].nunique(),
                     "mean_phishing": X.loc[y == 1, c].mean(),
                     "mean_legit": X.loc[y == 0, c].mean()})
    return pd.DataFrame(rows).sort_values("sep_auc", ascending=False).reset_index(drop=True)

auc_table = single_feature_auc(X_train, y_train)
auc_out = config.RESULTS_DIR / (f"results_sample_{SAMPLE}_single_feature_auc.csv" if SAMPLE else "single_feature_auc.csv")
auc_table.to_csv(auc_out, index=False)
auc_table.head(12).round(4)

In [ ]:
top = auc_table.head(20).iloc[::-1]
fig, ax = plt.subplots(figsize=(7, 6))
ax.barh(top["feature"], top["sep_auc"])
ax.axvline(0.95, color="red", ls="--", lw=1, label="0.95")
ax.set_xlim(0.5, 1.0)
ax.set_xlabel("single-feature ROC-AUC, max(AUC, 1 - AUC), training rows")
ax.set_title("How well each feature separates the classes on its own")
ax.legend(loc="lower right")
config.FIGURES_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(config.FIGURES_DIR / "rf_single_feature_auc.png", dpi=150, bbox_inches="tight")
plt.show()

**Reading the table:** `URLSimilarityIndex` alone reaches about 0.996. The next six (`LineOfCode`,
`NoOfExternalRef`, `NoOfImage`, `NoOfSelfRef`, `NoOfJS`, `NoOfCSS`) are page-content counts, each
around 0.96 to 0.99: in this dataset, phishing pages are almost empty (mean `LineOfCode` about 70
compared with about 1,900 for legitimate pages). That is probably because many phishing pages were
already taken down or were bare when the dataset was crawled. They are real signals, but they come
partly from how the dataset was collected.

In [ ]:
def class_constant_features(X, y):
    """Features that take one single value for every row of one class."""
    rows = []
    for c in X.columns:
        for cls, name in ((0, "legit"), (1, "phishing")):
            v = X.loc[y == cls, c]
            if v.nunique() == 1:
                val = v.iloc[0]
                rows.append({"feature": c, "constant_in": name, "value": val,
                             "other_class_share_same_value": (X.loc[y != cls, c] == val).mean()})
    return pd.DataFrame(rows)

display(class_constant_features(X_train, y_train).round(4))

# A one-line rule with no model: "URLSimilarityIndex below 100 means phishing".
from sklearn.metrics import confusion_matrix, f1_score
rule = (X_train["URLSimilarityIndex"] < 100).astype(int)
tn, fp, fn, tp = confusion_matrix(y_train, rule).ravel()
print(f"Rule USI < 100 on training rows: F1 {f1_score(y_train, rule):.4f}  (tn {tn}, fp {fp}, fn {fn}, tp {tp})")

**`URLSimilarityIndex` is 100 for every legitimate URL.** Under PhiUSIIL's definition, this
feature measures similarity to a list of known legitimate URLs, and the legitimate URLs come from
that same list. A threshold rule with no model reaches F1 of about 0.996 with zero false positives.
This is label leakage by construction: a new legitimate URL that is not on the list would not get
100, so the feature would not behave the same way in deployment.

`IsHTTPS` is also 1 for every legitimate URL (but only for half of the phishing ones), and every
legitimate URL has no `?`, `=`, `&`, IP address or obfuscation. So the legitimate class looks like
a set of clean HTTPS homepages. This is collection bias, not leakage, but it does make the task
easier than the real one.

In [ ]:
# Feature groups from the PhiUSIIL description: computed from the URL string vs from the fetched page.
URL_FEATURES = ["URLLength", "DomainLength", "IsDomainIP", "URLSimilarityIndex", "CharContinuationRate",
                "TLDLegitimateProb", "URLCharProb", "TLDLength", "NoOfSubDomain", "HasObfuscation",
                "NoOfObfuscatedChar", "ObfuscationRatio", "NoOfLettersInURL", "LetterRatioInURL",
                "NoOfDegitsInURL", "DegitRatioInURL", "NoOfEqualsInURL", "NoOfQMarkInURL",
                "NoOfAmpersandInURL", "NoOfOtherSpecialCharsInURL", "SpacialCharRatioInURL", "IsHTTPS"]
PAGE_FEATURES = [c for c in original if c not in URL_FEATURES]
assert set(URL_FEATURES) <= set(original)

strong = auc_table.loc[auc_table["sep_auc"] >= 0.95, "feature"].tolist()
without = lambda cols, drop: [c for c in cols if c not in drop]
variants = {
    "all original": original,
    "- URLSimilarityIndex": without(original, ["URLSimilarityIndex"]),
    f"- sep_auc >= 0.95 ({len(strong)})": without(original, strong),
    "URL only": URL_FEATURES,
    "URL only - USI": without(URL_FEATURES, ["URLSimilarityIndex"]),
    "URL only - USI - IsHTTPS": without(URL_FEATURES, ["URLSimilarityIndex", "IsHTTPS"]),
    "page only": PAGE_FEATURES,
}
ablation = {}
for name, cols in variants.items():
    r = evaluate.cv_evaluate(forest.build(), X_train[cols], y_train)
    ablation[name] = {"n_features": len(cols)} | {k: r[k] for k in SHOW}
ablation = pd.DataFrame(ablation).T
ablation.round(4)

**What the ablation shows:**
- Dropping `URLSimilarityIndex` costs very little, because the page-content counts still separate the
  classes. So the near-perfect score does not come from one leaking column alone.
- Even without all seven strongest features, CV F1 stays around 0.997. The dataset is genuinely easy
  for a forest once many weak signals are combined.
- With **URL features only** (what a URL-only risk scorer would see before fetching the page), removing
  `URLSimilarityIndex` and `IsHTTPS` brings F1 down to about 0.96. That is the more honest view of
  how hard the URL part of the task is.

In [ ]:
# Near-duplicate check: clean() removes repeated URLs, but the same domain can still be on both sides.
dom = df["Domain"]
in_train = dom.loc[X_test.index].isin(set(dom.loc[X_train.index]))
print(f"Test rows whose domain also appears in training: {in_train.mean():.2%} of {len(in_train)}")

About 4% of test URLs share a domain with a training URL. That is small, but a grouped split by
domain would remove it completely. That would be a change to `data.split()` (owner Madushanka D.M.C)
and needs group agreement.

### Phase 2 outcome (for the group to decide)

| Finding | Kind | Proposal for the group |
|---|---|---|
| `URLSimilarityIndex` = 100 for every legitimate URL; one-line rule F1 about 0.996 | Label leakage by construction | Drop it in `select_features()` |
| `IsHTTPS` = 1 for every legitimate URL | Collection bias | Keep, but report it; show results with and without it |
| Page-content counts (`LineOfCode`, `NoOf*Ref`, `NoOfImage`, `NoOfJS`, `NoOfCSS`) each have AUC of 0.96 or higher | Collection bias (bare or taken-down phishing pages) | Keep, but report a URL-only result next to the full one |
| About 4% of test domains also appear in training | Mild near-duplicate overlap | Optional: grouped split by domain |

The forest hyperparameters in `build()` stay provisional until the group decides on the feature list.
After that, rerun `forest.tune()` on the agreed features.

## Phase 3: Re-tune on the agreed features

**Group decision:** drop `URLSimilarityIndex` (label leakage by construction). Keep `IsHTTPS` and the
page-content counts (collection bias, not leakage), and report a URL-only result next to the full one.
Until `features.select_features()` (owner Madushan S.C) does the drop, it is applied by
`main.DROPPED_FEATURES`, so `main.py` and this notebook use the same list.

In [ ]:
from main import DROPPED_FEATURES, without_dropped

kept = without_dropped(original)
X_train, X_test, y_train, y_test = data.split(df, kept)
print(f"dropped {DROPPED_FEATURES}: {len(original)} -> {len(kept)} features")

search3, seconds = forest.tune(X_train, y_train, n_iter=20, scoring="f1")
print(f"Search took {seconds/60:.1f} min")
out = config.RESULTS_DIR / (f"results_sample_{SAMPLE}_forest_search_phase3.csv" if SAMPLE else "forest_search_phase3.csv")
pd.DataFrame(search3.cv_results_).to_csv(out, index=False)
forest.search_table(search3, top=8)

In [ ]:
# Tuned build() vs the phase 1 build(), and the URL-only view, on the agreed features (CV, training rows only).
def phase1_build():
    return RandomForestClassifier(n_estimators=100, max_depth=30, min_samples_leaf=8, max_features="sqrt",
                                  n_jobs=-1, random_state=config.RANDOM_STATE)

url_kept = without_dropped(URL_FEATURES)
phase3 = {
    "phase 1 build(), agreed features": evaluate.cv_evaluate(phase1_build(), X_train, y_train),
    "new build(), agreed features": evaluate.cv_evaluate(forest.build(), X_train, y_train),
    "new build(), URL only": evaluate.cv_evaluate(forest.build(), X_train[url_kept], y_train),
}
phase3 = pd.DataFrame(phase3).T[SHOW]
phase3.round(5)

**What changed and why.** Without `URLSimilarityIndex` the search is no longer flat. Two candidates tie
for best (CV F1 0.99956, std 0.00027): 500 trees, `min_samples_leaf=1`, `sqrt` features, `max_depth`
20 or 30. Depth 20 is taken as the more regularised of the two and written into `forest.build()`.
The phase 1 values (100 trees, leaf 8) drop to 0.99788, about 6 fold-standard-deviations lower: once the
leaking column is gone, the forest needs full-size trees to combine the weaker signals.

**URL-only view.** With URL features only (still including `IsHTTPS`), CV F1 is 0.99576. The page
counts add little on top of the URL features, so the score does not depend on one feature group alone.
The phase 2 result (URL only without `IsHTTPS`: about 0.96) remains the most honest view of the task.

All of these are still above 0.99 on a 20k sample. They describe this dataset, which is easier than
real-world phishing detection because of the collection bias found in phase 2.

## Phase 4: Risk score and calibration (proposal 4.11)

The risk score is the forest's phishing probability. To read it as a risk, it should be calibrated:
of all URLs scored about 0.8, about 80 % should be phishing. `risk.calibration_report()` compares
no calibration, sigmoid (Platt) and isotonic using **out-of-fold** scores on the training rows
(each calibrated model calibrates inside its own training folds), so the test set is not used.

- **Brier score**: mean squared error of the probability; lower is better.
- **Log loss**: punishes confident mistakes hardest.
- **ROC-AUC**: ranking only, so calibration should leave it almost unchanged.

In [ ]:
calib_table, oof = risk.calibration_report(forest.build(), X_train, y_train)
calib_table.round(5)

In [ ]:
fig = risk.plot_calibration(y_train, oof, filename="rf_calibration_curve.png",
                            title="Random Forest calibration (out-of-fold, training rows)")
plt.show()
for m, s in oof.items():
    print(f"{m:9s} URLs scored between 0.05 and 0.95: {int(((s > 0.05) & (s < 0.95)).sum())}")

### Risk bands

`risk.LOW_MAX` = 0.30 and `risk.HIGH_MIN` = 0.70. A band is only useful if its observed phishing rate
matches its name: Low should be almost all legitimate, High almost all phishing, and Medium the
genuinely uncertain URLs. `risk.band_table()` checks this on the out-of-fold scores.

In [ ]:
pd.concat({m: risk.band_table(y_train, s).set_index("band") for m, s in oof.items()}).round(4)

**Calibration choice: sigmoid.** The uncalibrated forest is underconfident: 646 training URLs get a
score between 0.05 and 0.95 although almost all of them are classified correctly (averaging 500 trees
pulls scores towards the middle). Sigmoid calibration cuts the Brier score from 0.00131 to 0.00043 and
log loss from 0.0086 to 0.0023, with ROC-AUC unchanged at 1.0. Isotonic is close on Brier score but worse
on log loss and slightly lowers ROC-AUC (it is a step function, so it creates ties); with only a handful of
uncertain URLs per fold it also has very little data to fit its steps. So `risk.CALIBRATION = "sigmoid"`.

**Bands: 0.30 / 0.70 kept.** With the calibrated score, Low is 99.96 % legitimate, High is 99.99 %
phishing and Medium (0.1 % of URLs) is about one-third phishing: the names match what the bands contain.

**Trade-off to note.** The uncalibrated score put every mistake into Medium (Low: 0 phishing, High: 0
legitimate), because its scores were less extreme. Calibration makes the score more accurate as a
probability, but 4 phishing URLs move into Low and 1 legitimate URL into High. If the group prefers
"never call a phishing URL Low" over probability accuracy, the alternative is to keep the raw score or
lower `LOW_MAX`; that would be a group choice, not a tuning result.

## Phase 5: Explainability (proposal 4.13)

SHAP `TreeExplainer` on the forest fitted on the training rows, for a sample of 1,000 held-out URLs.
Explanations do not choose any model or setting, so using held-out rows here does not affect tuning.
Values are in units of phishing probability; positive pushes towards phishing.

In [ ]:
rf = forest.train(X_train, y_train)
X_explain = X_test.iloc[:1000]
shap_rf = explain.contributions(rf, X_explain)
rf_importance = explain.rank_importance(X_explain.columns, shap_rf.values, "mean_abs_shap")
explain.summary_plot(rf, X_explain, "rf_shap_summary.png", values=shap_rf.values)
display(rf_importance.head(15).round(4))

# Share of total |SHAP| that comes from page-content features vs URL features.
share = rf_importance.assign(group=rf_importance["feature"].isin(URL_FEATURES).map({True: "URL", False: "page"}))
share.groupby("group")["mean_abs_shap"].sum() / share["mean_abs_shap"].sum()

In [ ]:
from IPython.display import Image
Image(config.FIGURES_DIR / "rf_shap_summary.png")

### Logistic Regression for comparison

A linear model is explained by its own coefficients: coefficient x standardised value is each
feature's contribution to the log-odds (what `shap.LinearExplainer` gives for independent features).
`logreg.build()` is Madushanka's tuned pipeline; it was tuned with `URLSimilarityIndex` still in, so
this view is provisional until LR is re-tuned on the agreed features.

In [ ]:
lr = logreg.train(X_train, y_train)
lr_importance = explain.linear_importance(lr, X_explain)
both = rf_importance.head(10)[["feature"]].assign(rf_rank=range(1, 11))
both["lr_rank"] = both["feature"].map({f: i + 1 for i, f in enumerate(lr_importance["feature"])})
display(lr_importance.head(10).round(4))
both

### Single URLs: risk score + level + predicted class + top features

The served score is the sigmoid-calibrated forest (`risk.calibrate`, training rows only). It is
explained by the uncalibrated forest's SHAP values: sigmoid calibration is monotonic, so it changes
the scale of the score but not which features push it up or down.

In [ ]:
risk_model = risk.calibrate(forest.build(), X_train, y_train)
scores = risk.risk_score(risk_model, X_explain)
per_url = explain.explain_predictions(rf, X_explain, scores, top=3, contrib=shap_rf)
per_url.insert(0, "URL", df.loc[X_explain.index, "URL"].values)
per_url.insert(1, "is_phishing", y_test.loc[X_explain.index].values)

examples = {
    "clear phishing": per_url[per_url["is_phishing"] == 1]["risk_score"].idxmax(),
    "clear legitimate": per_url[per_url["is_phishing"] == 0]["risk_score"].idxmin(),
    "most uncertain": (per_url["risk_score"] - 0.5).abs().idxmin(),
}
for name, i in examples.items():
    ex = explain.explain_one(rf, X_explain.loc[[i]], scores[X_explain.index.get_loc(i)], top=5)
    truth = "phishing" if per_url.loc[i, "is_phishing"] else "legitimate"
    print(f"{name}: {per_url.loc[i, 'URL']}  (true: {truth})")
    print(f"  risk score {ex.attrs['risk_score']:.3f} -> {ex.attrs['risk_level']}")
    display(ex.round(4))

**What drives the forest.** The top four SHAP features are page-content counts (`LineOfCode`,
`NoOfSelfRef`, `NoOfExternalRef`, `NoOfImage`), then `IsHTTPS`. Page features carry about 74 % of the
total |SHAP| and URL features about 26 %. This matches phase 2: the forest relies most on the page
signals that come partly from collection bias (bare or taken-down phishing pages).

**Logistic Regression looks at different things.** LR's top features are `IsHTTPS` and the URL
character ratios (`LetterRatioInURL`, `SpacialCharRatioInURL`, `DegitRatioInURL`); the forest's top page
counts rank only 7th to 18th for LR. A linear model cannot use thresholds such as "almost no lines of
code", so it leans on the URL string, while the forest picks up the strong non-linear page signals.

**Single URLs.** A clear phishing URL (a `web.app` subdomain with many digits and special characters
and a nearly empty page) scores 1.000 (High); a clear legitimate one (a large page with many self
references and images) scores 0.001 (Low). The most uncertain URL in the sample is legitimate but scores
0.457 (Medium), because its digits and special characters in the URL point to phishing while its page
content points to legitimate. This is the kind of URL the Medium band is for.

## Final test-set report

The held-out test is run once by `python main.py` with the frozen configurations (`reports/official_comparison.csv`,
`reports/risk_bands.csv`, `reports/risk_scores.csv`). This cell displays them.

In [ ]:
official = config.RESULTS_DIR / "official_comparison.csv"
if official.exists():
    display(pd.read_csv(official).set_index("model").T.round(5))
    display(pd.read_csv(config.RESULTS_DIR / "risk_bands.csv").round(4))
else:
    print("reports/official_comparison.csv not found: run `python main.py` on the full data first.")

## Experiment log

| Date | What changed | Why | CV or test | Key metrics (F1 / ROC-AUC / recall) | Commit |
|---|---|---|---|---|---|
| 2026-11-17 | Baseline RF (200 trees, sklearn defaults), original (50) and engineered (52) features | Starting point (4.9) | CV, 5 folds, 20k sample | orig 0.9999 / 1.0000 / 0.9997; eng 0.9999 / 1.0000 / 0.9999 | phase 1 |
| 2026-11-17 | RandomizedSearchCV, 20 candidates, scored on F1, original features | Tune RF hyperparameters on train only | CV, 20k sample | 6-way tie at F1 0.99993; all 20 within 0.0001; 1.5-3.1 min | phase 1 |
| 2026-11-17 | build(): 100 trees, max_depth 30, min_samples_leaf 8, sqrt | Cheapest, most regularised tied candidate | CV, 20k sample | orig 0.9999 / 1.0000 / 0.9999; eng 0.9999 / 1.0000 / 0.9997; 1 CV error in 16,000 | phase 1 |
| 2026-11-17 | Single-feature ROC-AUC + class-constant check, original features | Leakage check (phase 2) | Train rows, 20k sample, no model | USI sep_auc 0.996, =100 for all legit; rule USI<100 F1 0.9958; 6 page counts sep_auc 0.96-0.99 | phase 2 |
| 2026-11-17 | Ablation: drop USI / drop 7 strongest / URL-only / page-only | How much the score depends on suspect features | CV, 20k sample | -USI 0.9979 / 1.0000 / 0.9966; -7 strongest 0.9974; URL-only -USI -IsHTTPS 0.9639 / 0.9961 / 0.9523; page-only 0.9932 | phase 2 |
| 2026-11-19 | Group decision: drop `URLSimilarityIndex` (`main.DROPPED_FEATURES`); re-ran `forest.tune()` (20 candidates) | Phase 2 leakage; phase 1 values were tuned with the leaking column | CV, 20k sample | best 0.99956 / 1.0000 / 0.9993 (500 trees, depth 20, leaf 1, sqrt); phase 1 build() 0.99788 | phase 3 |
| 2026-11-19 | URL-only view with new build() (URL features minus USI, `IsHTTPS` kept) | Report a URL-only result next to the full one | CV, 20k sample | 0.99576 / 0.99899 / 0.99561 | phase 3 |
| 2026-11-19 | `risk.calibration_report()`: none vs sigmoid vs isotonic, out-of-fold on training rows | Is the probability usable as a risk score (4.11)? | CV (nested), 20k sample | Brier 0.00131 / 0.00043 / 0.00045; log loss 0.0086 / 0.0023 / 0.0034; sigmoid chosen | phase 4 |
| 2026-11-19 | `risk.band_table()` with bands 0.30 / 0.70 on sigmoid scores | Justify the risk bands | CV, 20k sample | Low 0.04 % phishing, Medium 33 % (15 URLs), High 99.99 %; raw score had 0 errors outside Medium | phase 4 |
| 2026-11-19 | SHAP on 1,000 held-out URLs; LR coefficient contributions; single-URL explanations | Explainability (4.13) | Explanation only, 20k sample | Top RF: LineOfCode, NoOfSelfRef, NoOfExternalRef, NoOfImage, IsHTTPS; page features 74 % of mean abs SHAP | phase 5 |
| 2026-11-24 | **Updated protocol (Part A):** group-aware split, grouped CV, `url_engineered`; bounded RF search on a 50,000-row group-preserving training subset (8 candidates, one-SE rule on leaves/tree) | Agreed methodology; earlier phases used the ungrouped split and page features | CV (subset) | depth 20, leaf 1, max_features 0.5, 300 trees: 0.99119 ± 0.00181 (subset) | `reports/tuning/rf_subset_search.csv` |
| 2026-11-24 | Full-data `main.py`: grouped CV + held-out test, format rule, grouped calibration, SHAP (500 URLs) | Final progress-evaluation numbers | CV (full train) + test | CV F1 0.9950 ± 0.0002; test F1 0.9950 / ROC-AUC 0.9988; format rule test F1 0.9948; Brier 0.0038 | `reports/official_comparison.csv` |